# Stage 1: Data pipeline — Common Voice accent classification
Cleaned dataset: `paramawattanakrai/common-voice-accent-downsampled`

Produces two feature sets from the **same split**:
1. **Mel-spectrograms** (2D) -> CNN / BiLSTM / Attention
2. **Whisper encoder embeddings** -> MLP (pooled vector) / BiLSTM / CNN / Attention (frame sequence)

Run on Kaggle (add the dataset as input, enable GPU) or locally (set `DATA_DIR`).

In [ ]:
# !pip install -q librosa soundfile transformers scikit-learn pandas tqdm joblib

## 0. Config

In [ ]:
import json, os
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm

# --- paths ---
DATA_DIR = Path('/kaggle/input')   # searched recursively for train.tsv; on a local machine point this at the dataset folder
_hits = list(DATA_DIR.rglob('train.tsv')) if DATA_DIR.exists() else []
assert _hits, f'train.tsv not found under {DATA_DIR} - add the dataset as Input or fix DATA_DIR'
DATA_DIR = _hits[0].parent
print('DATA_DIR =', DATA_DIR)
OUT_DIR  = Path('/kaggle/working/data') if Path('/kaggle/working').exists() else Path('data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- audio / features ---
SR = 16_000
DURATION = 6.0          # seconds, crop/pad every clip to this
N_MELS, N_FFT, HOP = 128, 1024, 256
WHISPER_MODEL = 'openai/whisper-small'   # d=768. 'openai/whisper-base' = 512 (lighter)
POOL = 4                # avg-pool time for whisper sequence (50 fps -> 12.5 fps)
SEED = 42
MIN_PER_CLASS = 0       # drop accents with fewer clips than this (0 = keep all)
LIMIT = None            # e.g. 200 to smoke-test the whole pipeline on a few clips

## 1. Load metadata (dataset ships with official train / dev / test TSVs)
Columns: `filename, text, up_votes, down_votes, age, gender, accent, duration, path`. We keep the dataset's own splits (no re-splitting), so results are comparable across Mel and Whisper.

In [ ]:
AUDIO_EXT = ('.mp3', '.wav', '.flac', '.ogg')
tsvs = {p.stem: p for p in DATA_DIR.rglob('*.tsv')}
print('tsv files:', {k: str(v.relative_to(DATA_DIR)) for k, v in tsvs.items()})

parts = []
for name, split in [('train', 'train'), ('dev', 'val'), ('test', 'test')]:
    d = pd.read_csv(tsvs[name], sep='	', quoting=3); d['split'] = split; parts.append(d)
meta = pd.concat(parts, ignore_index=True)
print(meta.shape); display(meta.head(3))

In [ ]:
PATH_COL, LABEL_COL = 'path', 'accent'

# index every audio file by name so we don't depend on folder layout
audio_index = {}
for p in DATA_DIR.rglob('*'):
    if p.suffix.lower() in AUDIO_EXT:
        audio_index[p.name] = p
print('audio files indexed:', len(audio_index))

meta['abs_path'] = meta[PATH_COL].map(lambda x: audio_index.get(Path(str(x)).name))
print('missing audio:', meta['abs_path'].isna().sum())
meta = meta.dropna(subset=['abs_path', LABEL_COL]).copy()
print('clips matched:', len(meta))
meta.groupby([LABEL_COL, 'split']).size().unstack()

## 2. Label map + save `splits.csv`

In [ ]:
if MIN_PER_CLASS:
    vc = meta[LABEL_COL].value_counts(); meta = meta[meta[LABEL_COL].isin(vc[vc >= MIN_PER_CLASS].index)]
if LIMIT:   # smoke test: small stratified subset per split
    meta = meta.groupby(['split', LABEL_COL], group_keys=False).apply(lambda g: g.head(max(1, LIMIT // 20)))

meta = meta.reset_index(drop=True)
classes = sorted(meta[LABEL_COL].unique()); lab2id = {c: i for i, c in enumerate(classes)}
meta['label'] = meta[LABEL_COL]; meta['label_id'] = meta[LABEL_COL].map(lab2id)

df = meta[['abs_path', 'label', 'label_id', 'split']].rename(columns={'abs_path': 'path'}); df['path'] = df['path'].astype(str)
df.to_csv(OUT_DIR/'splits.csv', index=False); json.dump(lab2id, open(OUT_DIR/'labels.json', 'w'), indent=2)
print(len(classes), 'classes'); df.groupby(['label', 'split']).size().unstack()

## 3. Audio loader (mono, 16 kHz, trim silence, crop/pad to fixed length)

In [ ]:
import librosa

def load_audio(path, duration=DURATION):
    y, _ = librosa.load(path, sr=SR, mono=True)
    if len(y): y, _ = librosa.effects.trim(y, top_db=30)
    n = int(SR * duration)
    if len(y) >= n:
        s = (len(y) - n) // 2
        return y[s:s+n], n
    return np.pad(y, (0, n - len(y))), len(y)      # (waveform, valid_samples)

w, v = load_audio(df.path[0]); print(w.shape, v / SR, 's valid')

## 4. Feature set 1 — log-Mel spectrograms

In [ ]:
from joblib import Parallel, delayed

def mel_one(path):
    y, _ = load_audio(path)
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, fmin=20, fmax=8000)
    return librosa.power_to_db(m, ref=1.0, top_db=80.0).astype(np.float16)

mel = np.stack(Parallel(n_jobs=-1)(delayed(mel_one)(p) for p in tqdm(df.path, desc='mel')))   # [N, n_mels, T]
tr = mel[(df.split == 'train').values].astype(np.float32)
mel_stats = {'mean': float(tr.mean()), 'std': float(tr.std())}      # TRAIN only -> no leakage
np.save(OUT_DIR/'mel.npy', mel); json.dump(mel_stats, open(OUT_DIR/'mel_stats.json','w'))
print(mel.shape, mel_stats)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(14, 3))
for a, i in zip(ax, np.random.RandomState(0).choice(len(df), 3, replace=False)):
    a.imshow(mel[i].astype(np.float32), origin='lower', aspect='auto'); a.set_title(df.label[i])
plt.tight_layout()

## 5. Feature set 2 — Whisper encoder embeddings
Saves both a pooled vector `[N, D]` (MLP) and an avg-pooled frame sequence `[N, T', D]` (BiLSTM/CNN/Attention).
Whisper pads input to 30 s, so we keep only the first `DURATION*50` frames. Needs GPU for reasonable speed.

In [ ]:
import torch
from transformers import WhisperFeatureExtractor, WhisperModel

dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', dev)
fe  = WhisperFeatureExtractor.from_pretrained(WHISPER_MODEL)
enc = WhisperModel.from_pretrained(WHISPER_MODEL).encoder.to(dev).eval()
if dev == 'cuda': enc.half()

FRAMES = int(DURATION * 50); FRAMES -= FRAMES % POOL
BATCH = 16
seqs, pools = [], []
paths = df.path.tolist()
for i in tqdm(range(0, len(paths), BATCH), desc='whisper'):
    wavs, valids = zip(*[load_audio(p) for p in paths[i:i+BATCH]])
    x = fe(list(wavs), sampling_rate=SR, return_tensors='pt').input_features.to(dev)
    if dev == 'cuda': x = x.half()
    with torch.no_grad():
        h = enc(x).last_hidden_state[:, :FRAMES].float()              # [B, FRAMES, D]
    for j, v in enumerate(valids):                                      # mean over non-padded frames only
        nv = max(1, min(FRAMES, int(v / SR * 50))); pools.append(h[j, :nv].mean(0).cpu().numpy())
    B, _, D = h.shape
    seqs.append(h.view(B, FRAMES // POOL, POOL, D).mean(2).cpu().numpy().astype(np.float16))

w_seq, w_pool = np.concatenate(seqs), np.stack(pools).astype(np.float32)
np.save(OUT_DIR/'whisper_seq.npy', w_seq); np.save(OUT_DIR/'whisper_pool.npy', w_pool)
print(w_seq.shape, w_pool.shape)

## 6. PyTorch Datasets (for the model stage)
Row `i` of every `.npy` corresponds to row `i` of `splits.csv`.

| feature | item shape | for |
|---|---|---|
| `mel` | `[1, 128, T]` | CNN (use `x.squeeze(0).T` -> `[T,128]` for BiLSTM/Attention) |
| `whisper_seq` | `[T', D]` | BiLSTM / CNN(1D) / Attention |
| `whisper_pool` | `[D]` | MLP |

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

def make_datasets(out_dir, feature):
    out_dir = Path(out_dir); d = pd.read_csv(out_dir/'splits.csv')
    X = np.load(out_dir/f'{feature}.npy', mmap_mode='r'); y = d.label_id.values
    st = json.load(open(out_dir/'mel_stats.json')) if feature == 'mel' else None
    class DS(Dataset):
        def __init__(s, idx): s.idx = idx
        def __len__(s): return len(s.idx)
        def __getitem__(s, k):
            i = s.idx[k]; x = np.asarray(X[i], dtype=np.float32)
            if st: x = ((x - st['mean']) / (st['std'] + 1e-6))[None]
            return torch.from_numpy(x), int(y[i])
    return {s: DS(np.where(d.split == s)[0]) for s in ('train', 'val', 'test')}

for feat in ['mel', 'whisper_seq', 'whisper_pool']:
    ds = make_datasets(OUT_DIR, feat); x, t = ds['train'][0]
    print(f"{feat:13s} item {tuple(x.shape)}  train/val/test = {[len(ds[s]) for s in ('train','val','test')]}")